# 04 — Inventory optimization

Notebook này biến dự báo nhu cầu thành một **mô phỏng chính sách tồn kho** cho Store 1. Để so sánh công bằng, ta dùng cùng các ngày holdout có cả nhãn thực tế và dự báo LSTM/baseline.

Rossmann là dữ liệu bán lẻ lịch sử; dataset không cung cấp đầy đủ lead time, biaya đặt hàng, biaya lưu kho hay giá trị mất doanh thu. Vì vậy các chi phí bên dưới là **giả định có thể chỉnh sửa**, không phải số liệu thực tế của Rossmann. Mục tiêu là xây dựng framework và phân tích độ nhạy, không tuyên bố tối ưu vận hành thực tế.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULT_DIR = ARTIFACT_DIR / "results"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
for folder in (RESULT_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

LOOKBACK = 30
STORE_ID = 1

from src.data import load_rossmann_data, chronological_split
from src.features import (
    create_calendar_features, prepare_features, get_feature_columns,
    fit_feature_scaler, transform_features, fit_target_scaler,
    transform_target, inverse_transform_target, make_sequences,
    DEFAULT_LOOKBACK
)

train_df, official_test_df = load_rossmann_data(DATA_DIR, store_id=STORE_ID)
train_df = train_df.sort_values("Date").reset_index(drop=True)
splits = chronological_split(train_df, train_ratio=0.70, val_ratio=0.15, holdout_ratio=0.15)
print(f"Store: {STORE_ID} | labeled rows: {len(train_df):,}")
print(f"Train:      {splits['train']['Date'].min().date()} -> {splits['train']['Date'].max().date()} ({len(splits['train'])})")
print(f"Validation: {splits['validation']['Date'].min().date()} -> {splits['validation']['Date'].max().date()} ({len(splits['validation'])})")
print(f"Holdout:    {splits['holdout']['Date'].min().date()} -> {splits['holdout']['Date'].max().date()} ({len(splits['holdout'])})")

## 1. Nạp dự báo và chuẩn bị chuỗi holdout

In [ ]:
lstm_path = RESULT_DIR / "lstm_holdout_predictions.csv"
if not lstm_path.exists():
    raise FileNotFoundError("Run 03_lstm.ipynb first to create lstm_holdout_predictions.csv")

lstm_df = pd.read_csv(lstm_path, parse_dates=["Date"])
baseline_path = RESULT_DIR / "baseline_holdout_predictions.csv"
if not baseline_path.exists():
    raise FileNotFoundError("Run 02_baseline.ipynb first to create baseline_holdout_predictions.csv")
baseline_df = pd.read_csv(baseline_path, parse_dates=["Date"])

eval_df = lstm_df.merge(
    baseline_df[["Date", "SeasonalNaive_7d", "MovingAverage_30d"]],
    on="Date", how="inner", validate="one_to_one"
).sort_values("Date").reset_index(drop=True)

# Rossmann Sales are zero on closed days. For inventory demand, a closed day
# naturally has zero realized sales; forecast values are floored at zero.
eval_df["LSTM_Prediction"] = eval_df["LSTM_Prediction"].clip(lower=0)
eval_df["SeasonalNaive_7d"] = eval_df["SeasonalNaive_7d"].clip(lower=0)
eval_df["MovingAverage_30d"] = eval_df["MovingAverage_30d"].clip(lower=0)

print("Evaluation days:", len(eval_df))
display(eval_df.head())

## 2. Chính sách (s, Q) và mô phỏng tồn kho

In [ ]:
# Adjustable assumptions — not provided by Rossmann dataset.
LEAD_TIME_DAYS = 3
REVIEW_PERIOD_DAYS = 1
INITIAL_INVENTORY = 250
ORDER_QUANTITY = 300
HOLDING_COST_PER_UNIT_DAY = 0.02
ORDERING_COST_PER_ORDER = 20.0
STOCKOUT_COST_PER_UNIT = 5.0
SERVICE_LEVEL_Z = 1.645  # approximate 95% one-sided target

def estimate_safety_stock(validation_actual, validation_forecast, lead_time, z=SERVICE_LEVEL_Z):
    """Approximate safety stock from validation residual std and lead time."""
    errors = np.asarray(validation_actual, float) - np.asarray(validation_forecast, float)
    errors = errors[np.isfinite(errors)]
    if len(errors) < 2:
        return 0.0
    daily_error_std = np.std(errors, ddof=1)
    return float(max(0.0, z * daily_error_std * np.sqrt(max(lead_time, 1))))

def simulate_s_q(actual_demand, forecast_demand, dates, safety_stock,
                 lead_time=LEAD_TIME_DAYS, initial_inventory=INITIAL_INVENTORY,
                 order_quantity=ORDER_QUANTITY, holding_cost=HOLDING_COST_PER_UNIT_DAY,
                 ordering_cost=ORDERING_COST_PER_ORDER, stockout_cost=STOCKOUT_COST_PER_UNIT):
    """
    Daily periodic review, order-up-to trigger approximated as (s,Q).
    Inventory position = on-hand + outstanding pipeline - backorders.
    Orders arrive after `lead_time` days. Lost sales are not backordered.
    """
    actual = np.maximum(np.asarray(actual_demand, float), 0)
    forecast = np.maximum(np.asarray(forecast_demand, float), 0)
    dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    if len(actual) != len(forecast) or len(actual) != len(dates):
        raise ValueError("actual, forecast and dates must have equal lengths")

    # Estimate expected demand during lead time from current forecast.
    on_hand = float(initial_inventory)
    pipeline = {}  # arrival_index -> quantity
    records = []
    total_order_cost = total_holding_cost = total_stockout_cost = 0.0

    for i, date in enumerate(dates):
        arrivals = pipeline.pop(i, 0.0)
        on_hand += arrivals

        demand = actual[i]
        fulfilled = min(on_hand, demand)
        lost_sales = max(0.0, demand - fulfilled)
        on_hand -= fulfilled

        total_holding_cost += on_hand * holding_cost
        total_stockout_cost += lost_sales * stockout_cost

        # Use the next lead-time forecast horizon for reorder threshold.
        future_slice = forecast[i:min(i + max(lead_time, 1), len(forecast))]
        expected_lead_demand = float(np.sum(future_slice))
        reorder_point = expected_lead_demand + safety_stock

        outstanding = sum(pipeline.values())
        inventory_position = on_hand + outstanding

        order_qty = 0.0
        if inventory_position <= reorder_point:
            order_qty = float(order_quantity)
            arrival_idx = i + max(lead_time, 1)
            if arrival_idx < len(actual):
                pipeline[arrival_idx] = pipeline.get(arrival_idx, 0.0) + order_qty
            total_order_cost += ordering_cost

        records.append({
            "Date": date, "Actual_Demand": demand, "Forecast_Demand": forecast[i],
            "On_Hand_End": on_hand, "Lost_Sales": lost_sales,
            "Arrivals": arrivals, "Inventory_Position": inventory_position,
            "Reorder_Point": reorder_point, "Order_Qty": order_qty,
            "Holding_Cost": on_hand * holding_cost,
            "Stockout_Cost": lost_sales * stockout_cost,
        })

    result = pd.DataFrame(records)
    summary = {
        "total_holding_cost": total_holding_cost,
        "total_ordering_cost": total_order_cost,
        "total_stockout_cost": total_stockout_cost,
        "total_cost": total_holding_cost + total_order_cost + total_stockout_cost,
        "average_on_hand": result["On_Hand_End"].mean(),
        "total_lost_sales": result["Lost_Sales"].sum(),
        "fill_rate_pct": 100 * (1 - result["Lost_Sales"].sum() / max(result["Actual_Demand"].sum(), 1)),
        "orders_placed": int((result["Order_Qty"] > 0).sum()),
    }
    return result, summary

def baseline_predictions(df):
    """Build causal baseline forecasts from earlier observed Sales."""
    out = df[["Date", "Store", "Sales", "Open", "Promo"]].copy()
    sales = df["Sales"].astype(float)
    out["Naive_1d"] = sales.shift(1)
    out["SeasonalNaive_7d"] = sales.shift(7)
    out["MovingAverage_7d"] = sales.shift(1).rolling(7, min_periods=7).mean()
    out["MovingAverage_30d"] = sales.shift(1).rolling(30, min_periods=30).mean()
    return out

# Estimate safety stock from validation data using the best available
# validation baseline (seasonal naive); this does not use holdout residuals.
val_start, val_end = splits["validation"]["Date"].min(), splits["validation"]["Date"].max()
val_base = baseline_predictions(train_df)
val_part = val_base.loc[val_base["Date"].between(val_start, val_end)].dropna(subset=["SeasonalNaive_7d"])
safety_stock = estimate_safety_stock(
    val_part["Sales"], val_part["SeasonalNaive_7d"], LEAD_TIME_DAYS
)
print(f"Estimated safety stock: {safety_stock:.1f} units (based on validation residuals)")

## 3. So sánh chính sách dùng các nguồn dự báo

In [ ]:
policies = {
    "LSTM": eval_df["LSTM_Prediction"].to_numpy(),
    "SeasonalNaive_7d": eval_df["SeasonalNaive_7d"].to_numpy(),
    "MovingAverage_30d": eval_df["MovingAverage_30d"].to_numpy(),
}
simulation_frames = {}
summaries = []

for name, forecast_values in policies.items():
    sim, summary = simulate_s_q(
        actual_demand=eval_df["Sales"].to_numpy(),
        forecast_demand=forecast_values,
        dates=eval_df["Date"],
        safety_stock=safety_stock,
    )
    sim["Policy"] = name
    simulation_frames[name] = sim
    summaries.append({"forecast_policy": name, **summary})

inventory_comparison = pd.DataFrame(summaries).sort_values("total_cost")
display(inventory_comparison)

## 4. Trực quan hóa và lưu kết quả

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(eval_df["Date"], eval_df["Sales"], label="Actual demand", linewidth=2)
axes[0].plot(eval_df["Date"], eval_df["LSTM_Prediction"], label="LSTM forecast", alpha=.8)
axes[0].set_ylabel("Units"); axes[0].legend(); axes[0].set_title("Demand on holdout")

for name, sim in simulation_frames.items():
    axes[1].plot(sim["Date"], sim["On_Hand_End"], label=name, alpha=.85)
axes[1].set_ylabel("Ending inventory"); axes[1].set_xlabel("Date")
axes[1].set_title("Simulated ending inventory"); axes[1].legend()
plt.tight_layout(); plt.show()

inventory_comparison.to_csv(RESULT_DIR / "inventory_policy_comparison.csv", index=False)
for name, sim in simulation_frames.items():
    safe_name = name.lower().replace("/", "_")
    sim.to_csv(RESULT_DIR / f"inventory_simulation_{safe_name}.csv", index=False)

print("Saved inventory results to", RESULT_DIR)

### Diễn giải và giới hạn
- Đây là mô phỏng đơn giản với lost sales, không phải mô hình đầy đủ của chuỗi cung ứng.
- Order quantity, lead time và các đơn giá là giả định. Nên chạy sensitivity analysis với nhiều giá trị.
- Reorder point được tính từ tổng forecast trong lead time cộng safety stock. Nếu muốn tối ưu đúng nghĩa, cần xác định hàm mục tiêu, miền tham số và tìm kiếm (s, Q) / policy trên validation, rồi chỉ báo cáo cuối cùng trên holdout.
- Một lần đặt hàng được tính phí ngay cả khi hàng về sau thời gian đánh giá; có thể bổ sung chi phí pipeline cuối kỳ nếu cần mô hình kế toán chi tiết hơn.